In [1]:
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 62.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 91.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.4/377.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.9 MB/s eta 0:00:00


In [2]:
import numpy as np
from math import log2, ceil

from qiskit import QuantumCircuit
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.quantum_info import SparsePauliOp


# --------- Helper: pad & normalize vectors ---------

def pad_and_normalize(vec):
    """
    Pad vec to next power-of-2 length and normalize.
    Returns (state, num_qubits).
    If vec is all zeros, returns (None, num_qubits).
    """
    v = np.asarray(vec, dtype=float)
    dim = len(v)
    if dim == 0:
        raise ValueError("Input vector has dimension 0.")

    num_qubits = ceil(log2(dim)) if dim > 0 else 0
    required_dim = 2**num_qubits

    if dim < required_dim:
        v = np.pad(v, (0, required_dim - dim), mode="constant", constant_values=0.0)

    norm = np.linalg.norm(v)
    if norm == 0:
        return None, num_qubits

    return v / norm, num_qubits


# --------- SWAP test circuit WITHOUT measurement ---------

def swap_test_circuit_from_vectors(v1, v2):
    """
    Build a SWAP test circuit from two classical vectors using amplitude encoding.
    Returns (qc, ancilla_index).

    This is your old qubit-by-qubit SWAP test, but:
    - no classical register,
    - no measurement,
    so it can be used with Estimator/NEAT.
    """
    # Pad + normalize
    v1_state_norm, nq1 = pad_and_normalize(v1)
    v2_state_norm, nq2 = pad_and_normalize(v2)

    if v1_state_norm is None or v2_state_norm is None:
        raise ValueError("One of the vectors is all zeros; cannot normalize.")

    if nq1 != nq2:
        raise ValueError("Vector dimensions (after padding) do not match.")

    num_qubits_vec = nq1
    total_qubits = 2 * num_qubits_vec + 1
    qc = QuantumCircuit(total_qubits, name="swap_test")

    # logical indices
    q1_l = list(range(num_qubits_vec))
    q2_l = list(range(num_qubits_vec, 2 * num_qubits_vec))
    ancilla = 2 * num_qubits_vec

    # State preparation (amplitude encoding)
    qc.initialize(v1_state_norm, q1_l)
    qc.initialize(v2_state_norm, q2_l)

    # SWAP-test core (same as your old code, just without measure)
    qc.h(ancilla)
    for i in range(num_qubits_vec):
        qc.cswap(ancilla, q1_l[i], q2_l[i])
    qc.h(ancilla)

    return qc, ancilla


# --------- Build an Estimator "pub" for this SWAP test ---------

def build_swap_pub_from_vectors(v1, v2):
    """
    Build a Primitive Unified Bloc (pub) suitable for EstimatorV2:
      pub = (circuit, [observable])

    Observable is Z on the ancilla qubit, encoded as a SparsePauliOp.
    """
    qc, ancilla = swap_test_circuit_from_vectors(v1, v2)
    n = qc.num_qubits

    # Pauli string with Z on ancilla, I everywhere else
    pauli_str = "I" * ancilla + "Z" + "I" * (n - ancilla - 1)
    observable = SparsePauliOp(pauli_str)

    pub = (qc, [observable])
    return pub


In [3]:
def ideal_fidelity_with_aer(v1, v2, precision=0.0):
    """
    Use qiskit_aer.primitives.EstimatorV2 to get an ideal estimate of:
      F = |⟨ψ|φ⟩|^2  (via ⟨Z_ancilla⟩)

    If precision=0.0, this is an exact statevector evaluation.
    If precision>0, Aer will sample N(expval, precision).
    """
    estimator = AerEstimator(options={"default_precision": precision})

    pub = build_swap_pub_from_vectors(v1, v2)
    job = estimator.run([pub])
    result = job.result()

    # result[0] is the PubResult; .data.evs is the array of expectation values
    # (one per observable in the pub). For us, it's a single Z on the ancilla.
    fidelity = result[0].data.evs[0]   # ⟨Z_ancilla⟩ = |⟨ψ|φ⟩|^2 for SWAP test

    # Convert to probability of measuring ancilla = 0, if you want to compare
    # directly with your old count-based implementation.
    p0 = (1 + fidelity) / 2.0

    return fidelity, p0


# --- quick smoke test ---

# if __name__ == "__main__":
    # Two simple vectors
v1 = [1, 0, 0, 0]
v2 = [1, 0, 0, 0]  # identical => fidelity ≈ 1

F, p0 = ideal_fidelity_with_aer(v1, v2)
print(f"Fidelity (ideal, Aer Estimator) = {F}")
print(f"P(ancilla=0) = {p0}")


Fidelity (ideal, Aer Estimator) = 1.0
P(ancilla=0) = 1.0


In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.debug_tools import Neat

def ideal_fidelity_with_neat(v1, v2, backend_name=None):
    """
    Same pub as before, but run through NEAT. This is essentially a
    wrapper over Aer EstimatorV2, using the IBM Runtime stack.
    """
    # Build the pub from your vectors
    pub = build_swap_pub_from_vectors(v1, v2)
    pubs = [pub]

    # Connect to IBM Quantum / Runtime
    service = QiskitRuntimeService()

    if backend_name is None:
        # pick an operational QPU (or you can choose a specific backend)
        backend = service.least_busy(operational=True, simulator=False)
    else:
        backend = service.backend(backend_name)

    analyzer = Neat(backend, noise_model=None)

    # (Optional) Cliffordize the pub for scalable classical sim
    # If your state prep uses non-Clifford gates, to_clifford() will map
    # to a nearby Clifford circuit with similar structure.
    clifford_pubs = analyzer.to_clifford(pubs)

    # Ideal (noiseless) simulation of expectation value(s)
    ideal_results = analyzer.ideal_sim(clifford_pubs, precision=0)

    # NeatResult is iterable; each entry is a NeatPubResult with .vals (array)
    fidelity = ideal_results[0].vals[0]   # single observable (ancilla Z)
    p0 = (1 + fidelity) / 2.0

    return fidelity, p0


Second part

In [5]:
import numpy as np
from math import ceil, log

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer import Aer


In [6]:
def pad_and_normalize(vec):
    """
    Pad vec to next power-of-2 length and normalize.
    Returns (state, num_qubits).
    If vec is all zeros, returns (None, num_qubits).
    """
    v = np.asarray(vec, dtype=float)
    dim = len(v)
    if dim == 0:
        raise ValueError("Input vector has dimension 0.")

    num_qubits = int(np.ceil(np.log2(dim)))
    required_dim = 2**num_qubits

    if dim < required_dim:
        v = np.pad(v, (0, required_dim - dim), mode="constant", constant_values=0.0)

    norm = np.linalg.norm(v)
    if norm == 0:
        return None, num_qubits

    return v / norm, num_qubits


def swap_test_circuit_from_vectors(v1, v2):
    """
    Build a SWAP test circuit from two classical vectors using amplitude encoding.
    Returns (qc, ancilla_index) with NO measurement (for Estimator/NEAT).
    """
    v1_state_norm, nq1 = pad_and_normalize(v1)
    v2_state_norm, nq2 = pad_and_normalize(v2)

    if v1_state_norm is None or v2_state_norm is None:
        raise ValueError("One of the vectors is all zeros; cannot normalize.")

    if nq1 != nq2:
        raise ValueError("Vector dimensions (after padding) do not match.")

    num_qubits_vec = nq1
    total_qubits = 2 * num_qubits_vec + 1
    qc = QuantumCircuit(total_qubits, name="swap_test")

    q1_l = list(range(num_qubits_vec))
    q2_l = list(range(num_qubits_vec, 2 * num_qubits_vec))
    ancilla = 2 * num_qubits_vec

    # State preparation (amplitude encoding)
    qc.initialize(v1_state_norm, q1_l)
    qc.initialize(v2_state_norm, q2_l)

    # SWAP test core (no measurement yet)
    qc.h(ancilla)
    for i in range(num_qubits_vec):
        qc.cswap(ancilla, q1_l[i], q2_l[i])
    qc.h(ancilla)

    return qc, ancilla


In [7]:
# def build_swap_pub_from_vectors(v1, v2):
#     """
#     Build a Primitive 'pub' = (circuit, [observable]) for EstimatorV2.
#     Observable is Z on the ancilla qubit.
#     """
#     qc, ancilla = swap_test_circuit_from_vectors(v1, v2)
#     n = qc.num_qubits

#     # Pauli string: Z on ancilla, I elsewhere
#     pauli_str = "I" * ancilla + "Z" + "I" * (n - ancilla - 1)
#     observable = SparsePauliOp(pauli_str)

#     pub = (qc, [observable])
#     return pub
def build_swap_pub_from_vectors(v1, v2):
    qc, ancilla = swap_test_circuit_from_vectors(v1, v2)
    n = qc.num_qubits

    # Qiskit: rightmost char -> qubit 0
    paulis = ["I"] * n
    paulis[n - 1 - ancilla] = "Z"   # put Z on ancilla
    pauli_str = "".join(paulis)

    observable = SparsePauliOp(pauli_str)
    pub = (qc, [observable])
    return pub


def ideal_p0_with_aer(v1, v2, estimator=None):
    """
    Get ideal P(ancilla=0) using Aer EstimatorV2 (noiseless).
    Estimator returns <Z_ancilla>, and P(0) = (1 + <Z>)/2.
    """
    if estimator is None:
        estimator = AerEstimator()

    pub = build_swap_pub_from_vectors(v1, v2)
    job = estimator.run([pub])
    result = job.result()

    z_expectation = result[0].data.evs[0]   # <Z_ancilla> = |<ψ|φ>|^2 for SWAP test
    p0_ideal = (1 + z_expectation) / 2.0

    return p0_ideal


In [8]:
def sample_p0_with_qasm(v1, v2, shots=1000, backend=None):
    """
    Run the SWAP test with actual measurements on the ancilla using qasm_simulator.
    Return empirical P_hat(ancilla=0) = counts('0') / shots.
    """
    if backend is None:
        backend = Aer.get_backend("qasm_simulator")

    qc, ancilla = swap_test_circuit_from_vectors(v1, v2)

    # Add a classical register and measure ONLY the ancilla
    c = ClassicalRegister(1, "c")
    qc.add_register(c)
    qc.measure(ancilla, c[0])

    qc_t = transpile(qc, backend)
    job = backend.run(qc_t, shots=shots)
    result = job.result()
    counts = result.get_counts()

    # Single classical bit => keys are '0' or '1'
    p0_hat = counts.get("0", 0) / shots
    return p0_hat


In [9]:
def hoeffding_shot_budget(epsilon, delta):
    """
    Hoeffding bound for X in [0,1]:
      P(|X_bar - E[X]| >= epsilon) <= 2 * exp(-2 * n * epsilon^2)
    Solve for n so the RHS <= delta.
    """
    return ceil((1 / (2 * epsilon**2)) * log(2 / delta))


def run_hoeffding_experiment(
    num_pairs=30,
    epsilon=0.05,
    delta=0.05,
    dim=4,
    seed=42,
):
    """
    1. Sample random classical vector pairs (dimension `dim`).
    2. For each pair, compute ideal P(ancilla=0) via Estimator (noiseless).
    3. Use Hoeffding to choose shots m(epsilon, delta).
    4. Run qasm_simulator with m shots, get P_hat(0).
    5. Count how often |P_hat(0) - P_ideal(0)| > epsilon.

    This gives an empirical 'violation rate' you can compare to delta.
    """
    rng = np.random.default_rng(seed)
    estimator = AerEstimator()
    backend = Aer.get_backend("qasm_simulator")

    m = hoeffding_shot_budget(epsilon, delta)
    print(f"Using Hoeffding shot budget m = {m} for ε = {epsilon}, δ = {delta}")

    violations = 0
    all_errors = []

    for k in range(num_pairs):
        # Random real vectors of given dimension
        v1 = rng.normal(size=dim)
        v2 = rng.normal(size=dim)

        # Ideal Bernoulli mean (P(ancilla=0))
        p0_ideal = ideal_p0_with_aer(v1, v2, estimator=estimator)

        # Sample-based estimate with m shots
        p0_hat = sample_p0_with_qasm(v1, v2, shots=m, backend=backend)

        err = abs(p0_hat - p0_ideal)
        all_errors.append(err)
        if err > epsilon:
            violations += 1

        print(f"Pair {k+1:02d}: p0_ideal={p0_ideal:.4f}, p0_hat={p0_hat:.4f}, |err|={err:.4f}")

    empirical_delta = violations / num_pairs
    print("\n=== Summary ===")
    print(f"Total pairs              : {num_pairs}")
    print(f"Violations (> ε)         : {violations}")
    print(f"Empirical violation rate : {empirical_delta:.3f} (compare to δ = {delta})")
    print(f"Max abs error            : {max(all_errors):.4f}")


In [10]:
run_hoeffding_experiment(
    num_pairs=30,
    epsilon=0.05,
    delta=0.05,
    dim=4,
    seed=123,
)

Using Hoeffding shot budget m = 738 for ε = 0.05, δ = 0.05
Pair 01: p0_ideal=0.8196, p0_hat=0.8157, |err|=0.0039
Pair 02: p0_ideal=0.5050, p0_hat=0.5352, |err|=0.0302
Pair 03: p0_ideal=0.7944, p0_hat=0.8293, |err|=0.0348
Pair 04: p0_ideal=0.6821, p0_hat=0.6680, |err|=0.0140
Pair 05: p0_ideal=0.6619, p0_hat=0.6721, |err|=0.0102
Pair 06: p0_ideal=0.5098, p0_hat=0.5515, |err|=0.0417
Pair 07: p0_ideal=0.5000, p0_hat=0.4648, |err|=0.0352
Pair 08: p0_ideal=0.5828, p0_hat=0.5854, |err|=0.0026
Pair 09: p0_ideal=0.9479, p0_hat=0.9499, |err|=0.0020
Pair 10: p0_ideal=0.5006, p0_hat=0.5108, |err|=0.0103
Pair 11: p0_ideal=0.6652, p0_hat=0.7060, |err|=0.0408
Pair 12: p0_ideal=0.7274, p0_hat=0.7385, |err|=0.0111
Pair 13: p0_ideal=0.8574, p0_hat=0.8821, |err|=0.0247
Pair 14: p0_ideal=0.6544, p0_hat=0.6789, |err|=0.0245
Pair 15: p0_ideal=0.7175, p0_hat=0.7222, |err|=0.0047
Pair 16: p0_ideal=0.5475, p0_hat=0.5420, |err|=0.0055
Pair 17: p0_ideal=0.5668, p0_hat=0.5759, |err|=0.0091
Pair 18: p0_ideal=0.541